In [1]:
import torch
import numpy as np

from src.train import evaluate_test_set
from src.datasets import H5Dataset

In [2]:
trial = 2
experiment = 'CPLS'

CSV_PATH_TEST = 'dataframes/annotations_all_HunCRC.csv'
H5_DIR_TEST = "features/features_conch_v15_HUN"
LABEL_COL = 'label'
ID_COL = 'slide'
model_dir = f"artifacts/{experiment}/trial_{trial}/models"
n_splits = 5


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


test_dataset = H5Dataset(
    csv_path=CSV_PATH_TEST,
    feats_path=H5_DIR_TEST,
    label_col=LABEL_COL,
    split='test',
    id_col=ID_COL
)

In [3]:
test_results = evaluate_test_set(
    test_dataset=test_dataset,
    model_dir=model_dir,
    device=device,
    n_splits=n_splits
)

"Model name abmil.base_mammoth.conch_v15 does not have a task, using default task none.
Auto-computed LoRA rank: 13 (from dimensions: input_dim=768, slot_dim=256, output_dim=512, num_experts=30)


C:\Users\gomaaad\Projects\CRC_ABMIL\millab\src\builders\ModelDict.py:193: UserWarning: Pretrained flag is True, but task is set to 'none'. Using random weights
  warnings.warn("Pretrained flag is True, but task is set to 'none'. Using random weights")


"Model name abmil.base_mammoth.conch_v15 does not have a task, using default task none.
Auto-computed LoRA rank: 13 (from dimensions: input_dim=768, slot_dim=256, output_dim=512, num_experts=30)


C:\Users\gomaaad\Projects\CRC_ABMIL\millab\src\builders\ModelDict.py:193: UserWarning: Pretrained flag is True, but task is set to 'none'. Using random weights
  warnings.warn("Pretrained flag is True, but task is set to 'none'. Using random weights")


"Model name abmil.base_mammoth.conch_v15 does not have a task, using default task none.
Auto-computed LoRA rank: 13 (from dimensions: input_dim=768, slot_dim=256, output_dim=512, num_experts=30)


C:\Users\gomaaad\Projects\CRC_ABMIL\millab\src\builders\ModelDict.py:193: UserWarning: Pretrained flag is True, but task is set to 'none'. Using random weights
  warnings.warn("Pretrained flag is True, but task is set to 'none'. Using random weights")


"Model name abmil.base_mammoth.conch_v15 does not have a task, using default task none.
Auto-computed LoRA rank: 13 (from dimensions: input_dim=768, slot_dim=256, output_dim=512, num_experts=30)


C:\Users\gomaaad\Projects\CRC_ABMIL\millab\src\builders\ModelDict.py:193: UserWarning: Pretrained flag is True, but task is set to 'none'. Using random weights
  warnings.warn("Pretrained flag is True, but task is set to 'none'. Using random weights")


"Model name abmil.base_mammoth.conch_v15 does not have a task, using default task none.
Auto-computed LoRA rank: 13 (from dimensions: input_dim=768, slot_dim=256, output_dim=512, num_experts=30)


C:\Users\gomaaad\Projects\CRC_ABMIL\millab\src\builders\ModelDict.py:193: UserWarning: Pretrained flag is True, but task is set to 'none'. Using random weights
  warnings.warn("Pretrained flag is True, but task is set to 'none'. Using random weights")


In [4]:
accs = test_results.get('accs', 0)
print(np.mean(accs))
accs

0.8475921825358622


[0.8275163666121113,
 0.8767919033407144,
 0.8163870703764321,
 0.8712428997785694,
 0.8460226725714837]

In [5]:
preds = test_results.get('preds', [])
truths = test_results.get('truths', [])

for idx, (pred, gt) in enumerate(zip(preds[0], truths[0])):
    print(f"Slide {idx}: Predicted={pred}, Ground Truth={gt}")

Slide 0: Predicted=0, Ground Truth=0
Slide 1: Predicted=2, Ground Truth=1
Slide 2: Predicted=3, Ground Truth=3
Slide 3: Predicted=2, Ground Truth=2
Slide 4: Predicted=3, Ground Truth=3
Slide 5: Predicted=0, Ground Truth=0
Slide 6: Predicted=3, Ground Truth=3
Slide 7: Predicted=0, Ground Truth=0
Slide 8: Predicted=1, Ground Truth=0
Slide 9: Predicted=3, Ground Truth=3
Slide 10: Predicted=0, Ground Truth=0
Slide 11: Predicted=2, Ground Truth=2
Slide 12: Predicted=0, Ground Truth=0
Slide 13: Predicted=1, Ground Truth=0
Slide 14: Predicted=0, Ground Truth=0
Slide 15: Predicted=2, Ground Truth=2
Slide 16: Predicted=1, Ground Truth=0
Slide 17: Predicted=3, Ground Truth=3
Slide 18: Predicted=0, Ground Truth=0
Slide 19: Predicted=1, Ground Truth=0
Slide 20: Predicted=0, Ground Truth=0
Slide 21: Predicted=0, Ground Truth=0
Slide 22: Predicted=0, Ground Truth=0
Slide 23: Predicted=3, Ground Truth=3
Slide 24: Predicted=1, Ground Truth=0
Slide 25: Predicted=0, Ground Truth=0
Slide 26: Predicted=2,

In [7]:
predictions = np.array(preds)
ground_truths = np.array(truths)

final_predictions = np.array([
    np.unique(predictions[:, i], return_counts=True)[0][
        np.argmax(np.unique(predictions[:, i], return_counts=True)[1])
    ]
    for i in range(predictions.shape[1])
])
ground_truths_int = ground_truths[0,:].astype(int)



# Calculate accuracy for each fold individually
fold_accuracies = []
for i in range(predictions.shape[0]):
    fold_pred = predictions[i, :]
    # Compare fold predictions to ground truth
    acc = np.mean(fold_pred == ground_truths_int)

    fold_accuracies.append(acc)

print(f" Accuracies for each fold: {fold_accuracies}")

# Compute Mean and Standard Deviation
mean_accuracy = np.mean(fold_accuracies)
std_accuracy = np.std(fold_accuracies)

# 3. Print the results
print(f"Results for Trial {trial}:")
print(f"Mean Accuracy: {mean_accuracy:.4f}")
print(f"Std Deviation: {std_accuracy:.4f}")

# Optional: Accuracy of the Majority Vote (Final Prediction)
final_acc = np.mean(final_predictions == ground_truths_int)
print(f"Majority Vote Accuracy: {final_acc:.4f}")

# Calculate accuracy per class per fold
num_classes = 4
class_labels = ['LGD', 'HGD', 'CRC', 'Others']

# Dictionary to store per-class accuracies for each fold
class_accuracies_per_fold = {i: [] for i in range(num_classes)}

for fold_idx in range(predictions.shape[0]):
    fold_pred = predictions[fold_idx, :]

    # Calculate accuracy for each class in this fold
    for class_id in range(num_classes):
        # Find all samples of this class
        class_mask = ground_truths_int == class_id

        if class_mask.sum() > 0:  # Only calculate if class exists in fold
            class_acc = np.mean(fold_pred[class_mask] == ground_truths_int[class_mask])
            class_accuracies_per_fold[class_id].append(class_acc)
        else:
            class_accuracies_per_fold[class_id].append(np.nan)

# Calculate mean and std deviation for each class
print("\n" + "="*60)
print("Per-Class Accuracy Statistics Across Folds:")
print("="*60)
for class_id in range(num_classes):
    accs = np.array(class_accuracies_per_fold[class_id])
    mean_acc = np.nanmean(accs)
    std_acc = np.nanstd(accs)
    print(f"{class_labels[class_id]:8} - Mean: {mean_acc*100:.2f}, Std: {std_acc*100:.2f}")
print("="*60)

 Accuracies for each fold: [np.float64(0.815), np.float64(0.865), np.float64(0.86), np.float64(0.84), np.float64(0.865)]
Results for Trial 2:
Mean Accuracy: 0.8490
Std Deviation: 0.0193
Majority Vote Accuracy: 0.8500

Per-Class Accuracy Statistics Across Folds:
LGD      - Mean: 81.49, Std: 4.29
HGD      - Mean: 73.00, Std: 15.68
CRC      - Mean: 96.47, Std: 4.32
Others   - Mean: 88.08, Std: 2.83
